# TouchTrace — Sensor Model Training (Colab)

Train the Phase 2 IMU LSTM + MDN on fused CSD4CA data (touch + accelerometer + gyroscope).
The existing `touch.onnx` stays frozen; this notebook only trains `sensor.onnx`.

**Before you start:** Runtime → Change runtime type → **GPU** (T4).

Repo: [github.com/ginwzy/TouchTrace](https://github.com/ginwzy/TouchTrace)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repository

Training data `sensor/sensor_data.jsonl.gz` (~43 MB, 32,027 swipes) is already in the repo (touch points with interpolated accel/gyro, no magnetometer).

In [ ]:
from pathlib import Path

REPO = "TouchTrace"
REPO_URL = "https://github.com/ginwzy/TouchTrace.git"
ROOT = Path("/content") / REPO
TRAIN_DIR = ROOT / "train"
SENSOR_DIR = TRAIN_DIR / "sensor"

if not ROOT.exists():
    !git clone {REPO_URL}
else:
    !git -C {ROOT} pull --ff-only

assert SENSOR_DIR.is_dir(), f"Missing directory: {SENSOR_DIR}"
%cd {TRAIN_DIR}

data = SENSOR_DIR / "sensor_data.jsonl.gz"
assert data.is_file(), f"Missing {data} (cwd={Path.cwd()}). If this clone is behind, upload the gz in the last cell."
print(f"Data: {data} ({data.stat().st_size / 1e6:.1f} MB)")
!ls -lh sensor/sensor_data.jsonl.gz

## 3. Install dependencies

In [ ]:
!pip install -q tensorflow tensorflow-probability tf-keras tf2onnx onnxruntime matplotlib pytest

## 4. Verify TensorFlow sees the GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", gpus)

if not gpus:
    print("\n⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU, then rerun from the top.")
else:
    print("\n✓ GPU ready.")

## 5. Data check (human IMU baseline)

Seated `|a|` should sit near 9.81 (gravity). Walking variance should be higher. Gyro near 0 when seated.

In [ ]:
!python -m sensor.eval

## 6. Train

Defaults (from `sensor/config.py`): 13-d input (previous IMU + remaining-frame step + condition), 6-d MDN output (accel + gyro), z-score on IMU, 3px path subsample, 250 max epochs, batch 256, prepad on GPU. No magnetometer, no geometric rotation of IMU.

| Option | Default | Description |
|--------|---------|-------------|
| `EPOCHS` | `None` | Max epochs; `None` uses config (250) |
| `LITE` | `False` | Use 2×64 LSTM instead of 2×128 |
| `SEQUENCE` | `False` | Mac Metal fallback only; Colab/CUDA use prepad |

In [ ]:
EPOCHS = None  # None → config default (250)
LITE = False
SEQUENCE = False

cmd = ["python", "-m", "sensor.train"]
if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LITE:
    cmd.append("--lite")
if SEQUENCE:
    cmd.append("--sequence")

print("Running:", " ".join(cmd))
!{" ".join(cmd)}

## 7. (Optional) Run unit tests

In [ ]:
!python -m pytest -q

## 8. Export ONNX

In [ ]:
LITE = globals().get("LITE", False)
export_cmd = "python -m sensor.convert"
if LITE:
    export_cmd += " --lite"

!{export_cmd}
!ls -lh sensor/sensor_model.h5 sensor/sensor_model_best.h5 sensor/sensor_model_last.h5 sensor/sensor_norm.json sensor/*.onnx 2>/dev/null || ls -lh sensor/sensor_model*.h5 sensor/sensor_norm.json

## 9. Download weights to your computer

In [ ]:
from google.colab import files
from pathlib import Path

LITE = globals().get("LITE", False)
downloads = ["sensor/sensor_model_best.h5", "sensor/sensor_model.h5", "sensor/sensor_norm.json"]
onnx = "sensor/sensor_lite.onnx" if LITE else "sensor/sensor.onnx"
if Path(onnx).exists():
    downloads.append(onnx)

for name in downloads:
    if Path(name).exists():
        print(f"Downloading {name} ...")
        files.download(name)
    else:
        print(f"Skip (not found): {name}")

---

### Using local code instead of GitHub

If your changes are not pushed yet, upload files into `train/sensor/` (`train.py`, `features.py`, `config.py`, `convert.py`, `eval.py`, and `sensor_data.jsonl.gz`).

In [ ]:
# Uncomment to upload local files into the current directory:
# from google.colab import files
# uploaded = files.upload()
# print("Uploaded:", list(uploaded.keys()))